In [1]:
import json
import os
import re
import subprocess
from pathlib import Path
from urllib.parse import urlparse

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

In [2]:
class InfrastructureAwarenessChecker:
    """
    InfrastructureAwarenessChecker

    This checker evaluates whether a research software artifact supports
    infrastructure-aware deployment.

    It checks whether the repository avoids unnecessary infrastructure burden,
    such as:
    - GPU-only or CUDA-only assumptions
    - hard-coded platform constraints
    - missing deployment/environment files
    - excessive dependency footprint
    - excessive repository size
    - hard-coded absolute paths
    - unnecessary privileged/container configuration

    Formal idea:
        infrastructureAwareness : (A, E) → {True, False}

    A = artifact repository
    E = encoded target environment and thresholds
    """

    def __init__(
        self,
        json_file,
        download_dir="downloads",
        default_max_repo_size_mb=1000,
        default_max_dependency_count=300,
        default_max_large_file_mb=100,
        minimum_score=5
    ):
        self.json_file = json_file
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)

        self.default_max_repo_size_mb = default_max_repo_size_mb
        self.default_max_dependency_count = default_max_dependency_count
        self.default_max_large_file_mb = default_max_large_file_mb
        self.minimum_score = minimum_score

        self.artifacts = self.load_metadata(json_file)
        self.results = []

    def load_metadata(self, json_file):
        with open(json_file, "r", encoding="utf-8") as file:
            data = json.load(file)

        return data.get("artifacts", {})

    def is_git_repository(self, uri):
        return isinstance(uri, str) and uri.startswith("https://github.com/")

    def repo_name_from_uri(self, uri):
        parsed = urlparse(uri)
        repo_name = parsed.path.rstrip("/").split("/")[-1]

        if repo_name.endswith(".git"):
            repo_name = repo_name[:-4]

        return repo_name or "repository"

    def clone_repository(self, artifact_id, uri):
        repo_name = self.repo_name_from_uri(uri)
        target_dir = self.download_dir / repo_name

        if target_dir.exists():
            print(f"📁 Repository already exists: {target_dir}")
            return target_dir

        print(f"⬇️ Cloning repository: {uri}")

        try:
            result = subprocess.run(
                ["git", "clone", "--depth", "1", uri, str(target_dir)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=120
            )

            if result.returncode != 0:
                print(f"❌ Failed to clone repository for {artifact_id}")
                print(result.stderr.strip())
                return None

            print(f"✅ Cloned to: {target_dir}")
            return target_dir

        except Exception as e:
            print(f"❌ Clone error for {artifact_id}: {e}")
            return None

    def get_directory_size_mb(self, directory):
        total_size = 0

        for root, dirs, files in os.walk(directory):
            for file in files:
                path = Path(root) / file
                try:
                    total_size += path.stat().st_size
                except OSError:
                    pass

        return total_size / (1024 * 1024)

    def count_files(self, directory):
        count = 0

        for _, _, files in os.walk(directory):
            count += len(files)

        return count

    def read_text_file(self, path, max_chars=200000):
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as file:
                return file.read(max_chars)
        except Exception:
            return ""

    def find_existing_files(self, repo_dir, possible_paths):
        found = []

        for relative_path in possible_paths:
            path = repo_dir / relative_path
            if path.exists():
                found.append(relative_path)

        return found

    def find_files_by_name(self, repo_dir, names):
        found = []
        names_lower = {name.lower() for name in names}

        for root, dirs, files in os.walk(repo_dir):
            for file in files:
                if file.lower() in names_lower:
                    try:
                        found.append(str((Path(root) / file).relative_to(repo_dir)))
                    except Exception:
                        found.append(str(Path(root) / file))

        return found

    def find_files_by_extensions(self, repo_dir, extensions):
        found = []

        for root, dirs, files in os.walk(repo_dir):
            for file in files:
                if any(file.lower().endswith(ext) for ext in extensions):
                    try:
                        found.append(Path(root) / file)
                    except Exception:
                        pass

        return found

    def collect_text_from_relevant_files(self, repo_dir):
        relevant_names = {
            "readme.md",
            "readme.rst",
            "readme.txt",
            "requirements.txt",
            "environment.yml",
            "environment.yaml",
            "pyproject.toml",
            "setup.py",
            "dockerfile",
            "docker-compose.yml",
            "docker-compose.yaml",
            "makefile",
            "tox.ini",
            "noxfile.py",
            "setup.cfg",
            "package.json",
            "pipfile"
        }

        files = []

        for root, dirs, filenames in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {".git", "__pycache__", ".mypy_cache", ".pytest_cache", "node_modules"}
            ]

            for filename in filenames:
                if filename.lower() in relevant_names:
                    files.append(Path(root) / filename)

        docs_dir = repo_dir / "docs"
        if docs_dir.exists():
            files.extend(self.find_files_by_extensions(docs_dir, [".md", ".rst", ".txt"]))

        combined = ""

        for path in files[:100]:
            try:
                relative = path.relative_to(repo_dir)
            except Exception:
                relative = path

            combined += f"\n\n--- FILE: {relative} ---\n\n"
            combined += self.read_text_file(path)

        return combined, files

    def parse_dependency_count(self, repo_dir):
        dependency_count = 0
        dependency_files = []

        requirements_path = repo_dir / "requirements.txt"
        if requirements_path.exists():
            dependency_files.append("requirements.txt")
            text = self.read_text_file(requirements_path)

            for line in text.splitlines():
                line = line.strip()
                if line and not line.startswith("#") and not line.startswith("-"):
                    dependency_count += 1

        environment_path = repo_dir / "environment.yml"
        if environment_path.exists():
            dependency_files.append("environment.yml")
            text = self.read_text_file(environment_path)

            for line in text.splitlines():
                line = line.strip()
                if line.startswith("- ") and not line.startswith("- pip"):
                    dependency_count += 1

        pyproject_path = repo_dir / "pyproject.toml"
        if pyproject_path.exists():
            dependency_files.append("pyproject.toml")
            text = self.read_text_file(pyproject_path)
            dependency_count += len(re.findall(r'"[^"]+>=?[^"]*"', text))

        setup_path = repo_dir / "setup.py"
        if setup_path.exists():
            dependency_files.append("setup.py")

        package_path = repo_dir / "package.json"
        if package_path.exists():
            dependency_files.append("package.json")
            text = self.read_text_file(package_path)
            dependency_count += len(re.findall(r'"[^"]+"\s*:', text))

        return dependency_count, dependency_files

    def find_large_files(self, repo_dir, max_large_file_mb):
        large_files = []

        for root, dirs, files in os.walk(repo_dir):
            dirs[:] = [
                d for d in dirs
                if d not in {".git", "__pycache__", ".mypy_cache", ".pytest_cache", "node_modules"}
            ]

            for file in files:
                path = Path(root) / file

                try:
                    size_mb = path.stat().st_size / (1024 * 1024)
                except OSError:
                    continue

                if size_mb > max_large_file_mb:
                    try:
                        relative = str(path.relative_to(repo_dir))
                    except Exception:
                        relative = str(path)

                    large_files.append({
                        "file": relative,
                        "size_mb": round(size_mb, 2)
                    })

        return large_files

    def detect_platform_constraints(self, text):
        text_lower = text.lower()

        patterns = {
            "gpu_cuda": [
                "cuda", "cudnn", "gpu required", "requires gpu", "nvidia",
                "tensorflow-gpu", "torch.cuda", "cupy"
            ],
            "os_specific": [
                "windows only", "linux only", "macos only", "ubuntu only",
                "requires linux", "requires windows", "requires macos"
            ],
            "absolute_paths": [
                "/home/", "/users/", "c:\\\\", "c:/", "/usr/local/",
                "/opt/", "/var/"
            ],
            "privileged_container": [
                "--privileged", "privileged: true", "sudo docker",
                "network_mode: host", "--net=host"
            ],
            "heavy_infrastructure": [
                "kubernetes", "slurm", "hpc", "cluster only",
                "requires cluster", "multi-gpu", "distributed training"
            ]
        }

        findings = {}

        for category, terms in patterns.items():
            found_terms = []

            for term in terms:
                if term in text_lower:
                    found_terms.append(term)

            findings[category] = found_terms

        return findings

    def evaluate_infrastructure_awareness(self, repo_dir, artifact_data):
        config = artifact_data.get("infrastructure_awareness", {})

        max_repo_size_mb = config.get("max_repo_size_mb", self.default_max_repo_size_mb)
        max_dependency_count = config.get("max_dependency_count", self.default_max_dependency_count)
        max_large_file_mb = config.get("max_large_file_mb", self.default_max_large_file_mb)
        minimum_score = config.get("minimum_score", self.minimum_score)
        allow_gpu_requirement = config.get("allow_gpu_requirement", False)
        allow_os_specific = config.get("allow_os_specific", False)
        allow_privileged_container = config.get("allow_privileged_container", False)

        repo_size_mb = self.get_directory_size_mb(repo_dir)
        file_count = self.count_files(repo_dir)
        dependency_count, dependency_files = self.parse_dependency_count(repo_dir)
        large_files = self.find_large_files(repo_dir, max_large_file_mb)

        combined_text, relevant_files = self.collect_text_from_relevant_files(repo_dir)
        platform_findings = self.detect_platform_constraints(combined_text)

        environment_files = self.find_existing_files(
            repo_dir,
            [
                "requirements.txt",
                "environment.yml",
                "environment.yaml",
                "pyproject.toml",
                "setup.py",
                "Pipfile",
                "poetry.lock",
                "package.json"
            ]
        )

        container_files = self.find_existing_files(
            repo_dir,
            [
                "Dockerfile",
                "docker-compose.yml",
                "docker-compose.yaml",
                ".devcontainer/devcontainer.json"
            ]
        )

        automation_files = self.find_existing_files(
            repo_dir,
            [
                "Makefile",
                "tox.ini",
                "noxfile.py",
                ".github/workflows",
                ".gitlab-ci.yml"
            ]
        )

        readme_files = self.find_existing_files(
            repo_dir,
            [
                "README.md",
                "README.rst",
                "README.txt",
                "readme.md",
                "readme.rst",
                "readme.txt"
            ]
        )

        docs_files = []
        if (repo_dir / "docs").exists():
            docs_files = self.find_files_by_extensions(repo_dir / "docs", [".md", ".rst", ".txt"])

        score = 0
        evidence = []
        issues = []

        if readme_files:
            score += 1
            evidence.append(f"README present: {', '.join(readme_files)}")
        else:
            issues.append("No README found")

        if environment_files:
            score += 1
            evidence.append(f"Environment/dependency specification present: {', '.join(environment_files)}")
        else:
            issues.append("No dependency/environment specification found")

        if container_files:
            score += 1
            evidence.append(f"Container/deployment specification present: {', '.join(container_files)}")
        else:
            issues.append("No container/deployment specification found")

        if automation_files:
            score += 1
            evidence.append(f"Automation/CI/build workflow evidence present: {', '.join(automation_files)}")
        else:
            issues.append("No automation/CI/build workflow evidence found")

        if docs_files:
            score += 1
            evidence.append(f"Documentation directory present with {len(docs_files)} text files")
        else:
            issues.append("No docs directory or deployment documentation found")

        if repo_size_mb <= max_repo_size_mb:
            score += 1
            evidence.append(f"Repository size within threshold: {repo_size_mb:.2f} MB <= {max_repo_size_mb} MB")
        else:
            issues.append(f"Repository size exceeds threshold: {repo_size_mb:.2f} MB > {max_repo_size_mb} MB")

        if dependency_count <= max_dependency_count:
            score += 1
            evidence.append(f"Dependency count within threshold: {dependency_count} <= {max_dependency_count}")
        else:
            issues.append(f"Dependency count exceeds threshold: {dependency_count} > {max_dependency_count}")

        if not large_files:
            score += 1
            evidence.append(f"No files larger than {max_large_file_mb} MB detected")
        else:
            issues.append(f"Large files detected: {large_files[:5]}")

        gpu_terms = platform_findings.get("gpu_cuda", [])
        os_terms = platform_findings.get("os_specific", [])
        absolute_path_terms = platform_findings.get("absolute_paths", [])
        privileged_terms = platform_findings.get("privileged_container", [])
        heavy_terms = platform_findings.get("heavy_infrastructure", [])

        if gpu_terms and not allow_gpu_requirement:
            issues.append(f"Possible GPU/CUDA specialization detected: {', '.join(gpu_terms[:8])}")
        else:
            score += 1
            evidence.append("No disallowed GPU/CUDA-only requirement detected")

        if os_terms and not allow_os_specific:
            issues.append(f"Possible OS-specific constraint detected: {', '.join(os_terms[:8])}")
        else:
            score += 1
            evidence.append("No disallowed OS-only constraint detected")

        if privileged_terms and not allow_privileged_container:
            issues.append(f"Possible privileged container setting detected: {', '.join(privileged_terms[:8])}")
        else:
            score += 1
            evidence.append("No privileged container requirement detected")

        if absolute_path_terms:
            issues.append(f"Possible hard-coded absolute paths detected: {', '.join(absolute_path_terms[:8])}")
        else:
            score += 1
            evidence.append("No obvious hard-coded absolute paths detected")

        if heavy_terms:
            issues.append(f"Possible heavy infrastructure dependency detected: {', '.join(heavy_terms[:8])}")
        else:
            score += 1
            evidence.append("No heavy infrastructure-only dependency detected")

        infrastructure_aware = score >= minimum_score and len([
            issue for issue in issues
            if "exceeds threshold" in issue
            or "GPU/CUDA" in issue
            or "OS-specific" in issue
            or "privileged" in issue
        ]) == 0

        return {
            "infrastructure_aware": infrastructure_aware,
            "score": score,
            "minimum_score": minimum_score,
            "repo_size_mb": round(repo_size_mb, 4),
            "file_count": file_count,
            "dependency_count": dependency_count,
            "dependency_files": dependency_files,
            "environment_files": environment_files,
            "container_files": container_files,
            "automation_files": automation_files,
            "readme_files": readme_files,
            "large_files": large_files,
            "platform_findings": platform_findings,
            "evidence": evidence,
            "issues": issues
        }

    def check_artifact(self, artifact_id, artifact_data):
        title = artifact_data.get("title", "")
        uri = artifact_data.get("uri", "")

        print("\n" + "=" * 80)
        print(f"🔍 Infrastructure Awareness Check for {artifact_id}")
        print(f"📦 Title: {title}")
        print(f"🔗 URI: {uri}")

        artifact_result = {
            "artifact_id": artifact_id,
            "title": title,
            "uri": uri,
            "infrastructure_aware": False,
            "status": "failed"
        }

        if not self.is_git_repository(uri):
            print("❌ Unsupported artifact type for this checker.")
            artifact_result["reason"] = "Unsupported artifact type."
            return artifact_result

        repo_dir = self.clone_repository(artifact_id, uri)

        if repo_dir is None:
            artifact_result["reason"] = "Repository could not be cloned."
            artifact_result["status"] = "not_evaluated_repository_unavailable"
            return artifact_result

        result = self.evaluate_infrastructure_awareness(repo_dir, artifact_data)
        artifact_result.update(result)

        print("\n📊 Infrastructure awareness evidence:")
        print(f" - Score: {result['score']} / required {result['minimum_score']}")
        print(f" - Repository size: {result['repo_size_mb']} MB")
        print(f" - File count: {result['file_count']}")
        print(f" - Dependency count: {result['dependency_count']}")
        print(f" - Environment files: {', '.join(result['environment_files']) if result['environment_files'] else 'None'}")
        print(f" - Container files: {', '.join(result['container_files']) if result['container_files'] else 'None'}")
        print(f" - Automation files: {', '.join(result['automation_files']) if result['automation_files'] else 'None'}")

        print("\n🔎 Evidence found:")
        if result["evidence"]:
            for item in result["evidence"]:
                print(f" - {item}")
        else:
            print(" - No evidence found.")

        print("\n⚠️ Issues / weak evidence:")
        if result["issues"]:
            for item in result["issues"]:
                print(f" - {item}")
        else:
            print(" - No major infrastructure issues detected.")

        if result["infrastructure_aware"]:
            artifact_result["status"] = "passed"
            print("\n✅ Infrastructure Awareness Result: PASSED")
        else:
            artifact_result["status"] = "failed"
            print("\n❌ Infrastructure Awareness Result: FAILED")

        return artifact_result

    def run(self):
        self.results = []

        print("🌱 Starting Infrastructure Awareness Fitness Function")
        print(f"📄 Metadata file: {self.json_file}")
        print(f"📁 Download directory: {self.download_dir}")

        for artifact_id, artifact_data in self.artifacts.items():
            result = self.check_artifact(artifact_id, artifact_data)
            self.results.append(result)

        print("\n" + "=" * 80)
        print("📌 Infrastructure Awareness Summary")
        print("=" * 80)

        for result in self.results:
            icon = "✅" if result["infrastructure_aware"] else "❌"
            print(f"{icon} {result['artifact_id']}: {result['status']}")

        return self.results

In [3]:
checker = InfrastructureAwarenessChecker(
    json_file="artifacts.json",
    download_dir="downloads",
    default_max_repo_size_mb=1000,
    default_max_dependency_count=300,
    default_max_large_file_mb=100,
    minimum_score=5
)

infrastructure_results = checker.run()

🌱 Starting Infrastructure Awareness Fitness Function
📄 Metadata file: artifacts.json
📁 Download directory: downloads

🔍 Infrastructure Awareness Check for artifact_1
📦 Title: We provide our resources in a dedicated repository
🔗 URI: https://github.com/hihey54/hicss58
📁 Repository already exists: downloads/hicss58

📊 Infrastructure awareness evidence:
 - Score: 10 / required 5
 - Repository size: 1.5754 MB
 - File count: 48
 - Dependency count: 3
 - Environment files: requirements.txt
 - Container files: None
 - Automation files: None

🔎 Evidence found:
 - README present: README.md
 - Environment/dependency specification present: requirements.txt
 - Repository size within threshold: 1.58 MB <= 1000 MB
 - Dependency count within threshold: 3 <= 300
 - No files larger than 100 MB detected
 - No disallowed GPU/CUDA-only requirement detected
 - No disallowed OS-only constraint detected
 - No privileged container requirement detected
 - No obvious hard-coded absolute paths detected
 - No hea

In [4]:
if PANDAS_AVAILABLE:
    df = pd.DataFrame(infrastructure_results)

    columns_to_show = [
        "artifact_id",
        "title",
        "infrastructure_aware",
        "status",
        "score",
        "minimum_score",
        "repo_size_mb",
        "file_count",
        "dependency_count",
        "environment_files",
        "container_files",
        "automation_files"
    ]

    existing_columns = [col for col in columns_to_show if col in df.columns]
    display(df[existing_columns])
else:
    for result in infrastructure_results:
        print(result)

,artifact_id,title,infrastructure_aware,status,score,minimum_score,repo_size_mb,file_count,dependency_count,environment_files,container_files,automation_files
0,artifact_1,We provide our resources in a dedicated reposi...,True,passed,10.0,5.0,1.5754,48.0,3.0,[requirements.txt],[],[]
1,artifact_2,Trending Customer Dataset,False,not_evaluated_repository_unavailable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,artifact_3,Python algorithms,False,failed,11.0,5.0,23.4867,1909.0,28.0,[pyproject.toml],[.devcontainer/devcontainer.json],[.github/workflows]
3,artifact_4,Scikit-learn,True,passed,12.0,5.0,45.0309,2504.0,59.0,[pyproject.toml],[.devcontainer/devcontainer.json],"[Makefile, .github/workflows]"
4,artifact_5,Pandas,True,passed,11.0,5.0,96.2912,4079.0,99.0,"[environment.yml, pyproject.toml]",[],[.github/workflows]
5,artifact_6,NumPy,True,passed,12.0,5.0,55.2295,2799.0,42.0,"[environment.yml, pyproject.toml]",[.devcontainer/devcontainer.json],[.github/workflows]
6,artifact_7,Matplotlib,True,passed,11.0,5.0,100.6347,4919.0,92.0,"[environment.yml, pyproject.toml]",[.devcontainer/devcontainer.json],"[tox.ini, .github/workflows]"
7,artifact_8,Scrapy,False,failed,10.0,5.0,7.2152,823.0,19.0,[pyproject.toml],[],"[tox.ini, .github/workflows]"
8,artifact_9,Flask,False,failed,10.0,5.0,2.8905,289.0,9.0,[pyproject.toml],[.devcontainer/devcontainer.json],[.github/workflows]
9,artifact_10,TensorFlow,False,failed,8.0,5.0,567.3861,39389.0,0.0,[],[],[.github/workflows]


In [5]:
output_file = "infrastructure_awareness_results.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(infrastructure_results, file, indent=4)

print(f"✅ Results saved to {output_file}")

✅ Results saved to infrastructure_awareness_results.json
